
# Waymo → corridor dataset builder

Цель ноутбука: собрать датасет для обучения модели вида

**(camera image + high-level condition) → future corridor mask / heatmap**

Ноутбук:
- использует только **GT trajectory**;
- не использует предсказания модели;
- работает с фактической структурой данных:
  - `precomputed_waymo_e2e/train/samples/...`
  - `precomputed_waymo_e2e/val/val_samples.jsonl`
- строит несколько target-режимов:
  - `corridor_mask`
  - `centerline_mask`
  - `gaussian_heatmap`


In [1]:
import io
import json
import math
import sys
import time
import traceback
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
from PIL import Image, ImageDraw, ImageFilter
from IPython.display import display


In [2]:
# === Пути ===
# Этот ноутбук пишет в НОВУЮ папку, чтобы не путаться со старыми прогонами.
# Все результаты будут лежать в: prepared_camera_corridor_dataset_filtered_front_v1/
#
# Что где искать после запуска:
# train manifest: prepared_camera_corridor_dataset_filtered_front_v1/train/FRONT/manifest.jsonl
# val manifest:   prepared_camera_corridor_dataset_filtered_front_v1/val/FRONT/manifest.jsonl
# logs:           prepared_camera_corridor_dataset_filtered_front_v1/_logs/
# progress json:  prepared_camera_corridor_dataset_filtered_front_v1/_logs/*_progress.json

PRECOMP_ROOT = Path("precomputed_waymo_e2e")
TRAIN_ROOT = PRECOMP_ROOT / "train"
VAL_ROOT = PRECOMP_ROOT / "val"
RAW_WAYMO_ROOT = Path(
    "/home/Jupyter/datasets/tesla/Waymo_open_dataset/waymo_open_dataset_end_to_end_camera_v_1_0_0"
)

OUT_ROOT = Path("prepared_camera_corridor_dataset_filtered_front_v1")
OUT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT = OUT_ROOT / "_logs"
LOG_ROOT.mkdir(parents=True, exist_ok=True)

print("PRECOMP_ROOT:", PRECOMP_ROOT.resolve(), "exists=", PRECOMP_ROOT.exists())
print("TRAIN_ROOT:", TRAIN_ROOT.resolve(), "exists=", TRAIN_ROOT.exists())
print("VAL_ROOT:", VAL_ROOT.resolve(), "exists=", VAL_ROOT.exists())
print("RAW_WAYMO_ROOT:", RAW_WAYMO_ROOT.resolve(), "exists=", RAW_WAYMO_ROOT.exists())
print("OUT_ROOT:", OUT_ROOT.resolve())
print("LOG_ROOT:", LOG_ROOT.resolve())


PRECOMP_ROOT: /home/Jupyter/datasets/tesla/Waymo_open_dataset/precomputed_waymo_e2e exists= True
TRAIN_ROOT: /home/Jupyter/datasets/tesla/Waymo_open_dataset/precomputed_waymo_e2e/train exists= True
VAL_ROOT: /home/Jupyter/datasets/tesla/Waymo_open_dataset/precomputed_waymo_e2e/val exists= True
RAW_WAYMO_ROOT: /home/Jupyter/datasets/tesla/Waymo_open_dataset/waymo_open_dataset_end_to_end_camera_v_1_0_0 exists= True
OUT_ROOT: /home/Jupyter/datasets/tesla/Waymo_open_dataset/prepared_camera_corridor_dataset_filtered_front_v1
LOG_ROOT: /home/Jupyter/datasets/tesla/Waymo_open_dataset/prepared_camera_corridor_dataset_filtered_front_v1/_logs


In [3]:

# === Импорты Waymo ===
# Путь взят по аналогии с твоим photos.ipynb.

WAYMO_SRC = Path("/home/Jupyter/datasets/tesla/Waymo_open_dataset/waymo-od/src")
WAYMO_SRC_STR = str(WAYMO_SRC)
if WAYMO_SRC_STR not in sys.path:
    sys.path.insert(0, WAYMO_SRC_STR)

from waymo_open_dataset import dataset_pb2
from waymo_open_dataset.protos import end_to_end_driving_data_pb2
import tensorflow as tf

print("tensorflow:", tf.__version__)
print("WAYMO_SRC exists:", WAYMO_SRC.exists())


2026-04-21 21:58:35.319608: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-21 21:58:35.428541: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /opt/nvshmem/lib:/usr/local/cuda/compat/lib:/usr/local/nvidia/lib:/usr/local/nvidia/lib64
2026-04-21 21:58:35.428596: I tensorflow/compiler/xla/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2026-04-21 21:58:36.071969: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 

tensorflow: 2.11.0
WAYMO_SRC exists: True


In [4]:

# === Загрузка split rows ===
# train: из train/samples/*/meta.json
# val:   из val/val_samples.jsonl


def load_jsonl_rows(path: Path) -> List[dict]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def load_train_rows_from_samples(train_root: Path) -> List[dict]:
    samples_root = train_root / "samples"
    if not samples_root.exists():
        raise FileNotFoundError(f"Train samples dir not found: {samples_root}")

    rows = []
    for sample_dir in sorted(samples_root.iterdir()):
        if not sample_dir.is_dir():
            continue
        if sample_dir.name.startswith("."):
            continue
        if sample_dir.name == "__pycache__":
            continue

        meta_path = sample_dir / "meta.json"
        if not meta_path.exists():
            continue

        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        row = {
            "split": "train",
            "sample_dir": str(sample_dir.resolve()),  # абсолютный путь
        }
        row.update(meta)
        rows.append(row)

    return rows


val_rows = load_jsonl_rows(VAL_ROOT / "val_samples.jsonl")
train_rows = load_train_rows_from_samples(TRAIN_ROOT)

print("train_rows:", len(train_rows))
print("val_rows:", len(val_rows))
print("train example keys:", sorted(train_rows[0].keys())[:20] if train_rows else None)
print("val example keys:", sorted(val_rows[0].keys())[:20] if val_rows else None)


train_rows: 415663
val_rows: 106360
train example keys: ['context_name', 'depth_path', 'future_xy', 'intent', 'lane_path', 'lane_prob_path', 'past_xy', 'record_idx', 'rgb_path', 'road_path', 'sample_dir', 'sample_id', 'shard_id', 'shard_path', 'split', 'timestamp_micros', 'veh_path']
val example keys: ['context_name', 'future_len', 'intent', 'past_len', 'sample_dir', 'sample_id', 'split']


In [5]:

# === Быстрый просмотр примеров row ===
print("TRAIN row example:")
print(train_rows[0])
print("VAL row example:")
print(val_rows[0])


TRAIN row example:
{'split': 'train', 'sample_dir': '/home/Jupyter/datasets/tesla/Waymo_open_dataset/precomputed_waymo_e2e/train/samples/00000000', 'sample_id': 0, 'context_name': 'afd7afd2fd7fe02b65210917a8b65da5-199', 'record_idx': 0, 'shard_id': 0, 'shard_path': '/home/Jupyter/datasets/tesla/Waymo_open_dataset/waymo_open_dataset_end_to_end_camera_v_1_0_0/training_202504031202_202504151040.tfrecord-00000-of-00263', 'timestamp_micros': 0, 'intent': 1, 'past_xy': [[-22.654296875, 0.1484375], [-21.794921875, 0.1484375], [-20.845703125, 0.14599609375], [-19.802734375, 0.138671875], [-18.65625, 0.13037109375], [-17.404296875, 0.1201171875], [-16.044921875, 0.10986328125], [-14.591796875, 0.09619140625], [-13.021484375, 0.07958984375], [-11.35546875, 0.06005859375], [-9.619140625, 0.03955078125], [-7.875, 0.02099609375], [-5.919921875, 0.0078125], [-4.05078125, -0.00146484375], [-2.00390625, -0.001953125], [0.0, 0.0]], 'future_xy': [[2.0390625, 0.00732421875], [4.11328125, 0.01806640625], 

In [6]:

# === Вспомогательные функции для пути sample_dir и meta ===


def resolve_sample_dir(row: dict, split_root: Optional[Path] = None) -> Path:
    sample_dir = Path(row["sample_dir"])
    if sample_dir.exists():
        return sample_dir

    if split_root is not None:
        candidate = split_root / row["sample_dir"]
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Could not resolve sample_dir: {row['sample_dir']}")


def load_meta_from_row(row: dict, split_root: Optional[Path] = None) -> Tuple[Path, dict]:
    sample_dir = resolve_sample_dir(row, split_root=split_root)
    meta_path = sample_dir / "meta.json"
    if not meta_path.exists():
        raise FileNotFoundError(meta_path)
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    return sample_dir, meta


In [7]:

# === Waymo raw record loading ===


def resolve_tfrecord_path(meta: dict, raw_root: Path) -> Path:
    shard_path = Path(meta["shard_path"])
    candidates = [
        shard_path,
        raw_root / shard_path,
        raw_root / shard_path.name,
    ]
    for c in candidates:
        if Path(c).exists():
            return Path(c)

    matches = list(raw_root.rglob(shard_path.name))
    if len(matches) == 0:
        raise FileNotFoundError(shard_path)
    return matches[0]


def load_e2ed_record(meta: dict, raw_root: Path):
    tfrecord_path = resolve_tfrecord_path(meta, raw_root)
    dataset = tf.data.TFRecordDataset(str(tfrecord_path), compression_type="")
    record_idx = int(meta["record_idx"])
    for i, raw in enumerate(dataset):
        if i == record_idx:
            e2e = end_to_end_driving_data_pb2.E2EDFrame()
            e2e.ParseFromString(raw.numpy())
            return tfrecord_path, e2e
    raise IndexError(record_idx)


In [8]:

# === Камеры и калибровка ===


def cam_name_to_str(cam_name_int):
    try:
        return dataset_pb2.CameraName.Name.Name(cam_name_int)
    except Exception:
        return str(cam_name_int)


def get_camera_image_and_calib(frame, camera_name):
    image_obj = None
    calib_obj = None

    for img in frame.images:
        if img.name == camera_name:
            image_obj = img
            break

    for calib in frame.context.camera_calibrations:
        if calib.name == camera_name:
            calib_obj = calib
            break

    if image_obj is None:
        raise ValueError(f"Camera image not found: {cam_name_to_str(camera_name)}")
    if calib_obj is None:
        raise ValueError(f"Camera calibration not found: {cam_name_to_str(camera_name)}")

    rgb = np.array(Image.open(io.BytesIO(image_obj.image)).convert("RGB"))
    return rgb, image_obj, calib_obj


In [9]:

# === Проекция vehicle-frame trajectory -> image plane ===
# Логика перенесена из photos.ipynb.


def invert_transform(T: np.ndarray) -> np.ndarray:
    return np.linalg.inv(T)


def get_vehicle_from_camera(calib_obj) -> np.ndarray:
    return np.array(calib_obj.extrinsic.transform, dtype=np.float64).reshape(4, 4)


def get_camera_from_vehicle(calib_obj) -> np.ndarray:
    return invert_transform(get_vehicle_from_camera(calib_obj))


def transform_points(T: np.ndarray, pts_xyz: np.ndarray) -> np.ndarray:
    pts_xyz = np.asarray(pts_xyz, dtype=np.float64)
    ones = np.ones((len(pts_xyz), 1), dtype=np.float64)
    pts_h = np.concatenate([pts_xyz, ones], axis=1)
    out = (T @ pts_h.T).T
    return out[:, :3]

def show_debug_grid_pil(rgb, centerline_mask, corridor_mask, overlay, scale=1.0):
    gap = 20
    h, w = rgb.shape[:2]

    rgb_pil = Image.fromarray(rgb.astype(np.uint8)).convert("RGB")
    center_pil = Image.fromarray(centerline_mask.astype(np.uint8)).convert("L").convert("RGB")
    corridor_pil = Image.fromarray(corridor_mask.astype(np.uint8)).convert("L").convert("RGB")
    overlay_pil = Image.fromarray(overlay.astype(np.uint8)).convert("RGB")

    canvas = Image.new("RGB", (w * 4 + gap * 3, h), (255, 255, 255))
    canvas.paste(rgb_pil, (0, 0))
    canvas.paste(center_pil, (w + gap, 0))
    canvas.paste(corridor_pil, (2 * (w + gap), 0))
    canvas.paste(overlay_pil, (3 * (w + gap), 0))

    if scale != 1.0:
        new_size = (int(canvas.size[0] * scale), int(canvas.size[1] * scale))
        canvas = canvas.resize(new_size, Image.Resampling.BILINEAR)

    display(canvas)

def project_vehicle_points_to_image(pts_vehicle_xyz: np.ndarray, calib_obj):
    pts_cam = transform_points(get_camera_from_vehicle(calib_obj), pts_vehicle_xyz)

    x = pts_cam[:, 0]
    y = pts_cam[:, 1]
    z = pts_cam[:, 2]

    valid = x > 1e-6

    u = np.full_like(x, np.nan, dtype=np.float64)
    v = np.full_like(x, np.nan, dtype=np.float64)

    xn = -y[valid] / x[valid]
    yn = -z[valid] / x[valid]

    intr = np.array(calib_obj.intrinsic, dtype=np.float64)
    fu, fv, cu, cv = intr[:4]
    k1, k2, p1, p2, k3 = intr[4:9]

    r2 = xn * xn + yn * yn
    radial = 1.0 + k1 * r2 + k2 * r2 * r2 + k3 * r2 * r2 * r2
    x_tan = 2 * p1 * xn * yn + p2 * (r2 + 2 * xn * xn)
    y_tan = p1 * (r2 + 2 * yn * yn) + 2 * p2 * xn * yn

    xd = xn * radial + x_tan
    yd = yn * radial + y_tan

    u[valid] = fu * xd + cu
    v[valid] = fv * yd + cv

    uv = np.stack([u, v], axis=1).astype(np.float32)
    return uv, valid, pts_cam


def traj_xy_to_vehicle_xyz(traj_xy: np.ndarray, z_vehicle: float = 0.0) -> np.ndarray:
    traj_xy = np.asarray(traj_xy, dtype=np.float32)
    if traj_xy.ndim != 2 or traj_xy.shape[1] != 2:
        raise ValueError(f"Expected [T,2], got {traj_xy.shape}")
    z = np.full((len(traj_xy), 1), float(z_vehicle), dtype=np.float32)
    return np.concatenate([traj_xy, z], axis=1)


def clip_uv_to_image(uv: np.ndarray, image_shape: Tuple[int, int, int]) -> np.ndarray:
    h, w = image_shape[:2]
    uv = np.asarray(uv, dtype=np.float32)
    inside = (
        np.isfinite(uv[:, 0]) & np.isfinite(uv[:, 1]) &
        (uv[:, 0] >= 0) & (uv[:, 0] < w) &
        (uv[:, 1] >= 0) & (uv[:, 1] < h)
    )
    return uv[inside]


def project_traj_xy(traj_xy: np.ndarray, calib_obj, image_shape, z_vehicle: float = 0.0):
    traj_xyz = traj_xy_to_vehicle_xyz(traj_xy, z_vehicle=z_vehicle)
    uv, valid, pts_cam = project_vehicle_points_to_image(traj_xyz, calib_obj)
    uv = clip_uv_to_image(uv[valid], image_shape)
    return uv, pts_cam[valid]


In [10]:

# === GT future trajectory extraction ===
# ВАЖНО:
# Ниже стоит максимально терпимая логика поиска GT в sample_dir.
# Если у тебя в исходном photos.ipynb GT брался из строго определенного файла,
# замени эту функцию на тот вариант один в один.

GT_CANDIDATE_FILENAMES = [
    "gt_future_xy.npy",
    "gt.npy",
    "future_xy.npy",
    "traj_gt.npy",
]


def extract_gt_future_xy(sample_dir: Path, row: dict, meta: dict) -> np.ndarray:
    # 1) Ищем прямой .npy файл в sample_dir
    for name in GT_CANDIDATE_FILENAMES:
        p = sample_dir / name
        if p.exists():
            arr = np.load(p)
            arr = np.asarray(arr)
            if arr.ndim == 2 and arr.shape[1] >= 2:
                return arr[:, :2].astype(np.float32)

    # 2) Ищем путь в meta.json
    meta_keys = [
        "gt_path", "gt_future_path", "future_xy_path", "traj_gt_path"
    ]
    for k in meta_keys:
        if k in meta:
            p = Path(meta[k])
            if not p.is_absolute():
                p = sample_dir / p
            if p.exists():
                arr = np.load(p)
                arr = np.asarray(arr)
                if arr.ndim == 2 and arr.shape[1] >= 2:
                    return arr[:, :2].astype(np.float32)

    # 3) Ищем массив прямо в row/meta
    direct_keys = [
        "gt", "future_xy", "traj_gt", "gt_future_xy"
    ]
    for src in (row, meta):
        for k in direct_keys:
            if k in src:
                arr = np.asarray(src[k])
                if arr.ndim == 2 and arr.shape[1] >= 2:
                    return arr[:, :2].astype(np.float32)

    raise FileNotFoundError(
        f"Could not find GT future trajectory for sample_dir={sample_dir}"
    )


In [11]:

# === Target generation ===


def _draw_polyline_mask(size_hw: Tuple[int, int], uv: np.ndarray, width: int) -> np.ndarray:
    h, w = size_hw
    canvas = Image.new("L", (w, h), 0)
    draw = ImageDraw.Draw(canvas)
    pts = [tuple(map(float, p)) for p in np.asarray(uv)]
    if len(pts) >= 2:
        draw.line(pts, fill=255, width=int(width), joint="curve")
    elif len(pts) == 1:
        x, y = pts[0]
        r = max(1, int(width // 2))
        draw.ellipse((x-r, y-r, x+r, y+r), fill=255)
    return np.array(canvas, dtype=np.uint8)


def make_centerline_mask(uv: np.ndarray, image_shape, line_width: int = 3) -> np.ndarray:
    h, w = image_shape[:2]
    return _draw_polyline_mask((h, w), uv, width=line_width)


def make_corridor_mask(uv: np.ndarray, image_shape, corridor_width: int = 17) -> np.ndarray:
    h, w = image_shape[:2]
    return _draw_polyline_mask((h, w), uv, width=corridor_width)


def make_gaussian_heatmap_from_corridor(corridor_mask: np.ndarray, sigma_px: float = 7.0) -> np.ndarray:
    pil = Image.fromarray(corridor_mask)
    blurred = pil.filter(ImageFilter.GaussianBlur(radius=float(sigma_px)))
    arr = np.array(blurred, dtype=np.float32)
    if arr.max() > 0:
        arr = arr / arr.max()
    return arr


In [12]:

# === Debug: проверка одного sample на train и val ===
CAMERA_NAME = dataset_pb2.CameraName.FRONT
Z_VEHICLE = 0.0


CAMERA_NAME = dataset_pb2.CameraName.FRONT
Z_VEHICLE = 0.0

def polyline_length_px(uv: np.ndarray) -> float:
    if uv is None or len(uv) < 2:
        return 0.0
    diffs = uv[1:] - uv[:-1]
    return float(np.linalg.norm(diffs, axis=1).sum())


def debug_one_row(
    row: dict,
    split_root: Optional[Path] = None,
    title: str = "",
    corridor_width: int = 17,
    centerline_width: int = 3,
    gaussian_sigma_px: float = 7.0,
):
    sample_dir, meta = load_meta_from_row(row, split_root=split_root)
    gt_xy = extract_gt_future_xy(sample_dir, row, meta)
    tfrecord_path, e2e = load_e2ed_record(meta, RAW_WAYMO_ROOT)
    rgb, image_obj, calib_obj = get_camera_image_and_calib(e2e.frame, CAMERA_NAME)
    uv_gt, _ = project_traj_xy(gt_xy, calib_obj, rgb.shape, z_vehicle=Z_VEHICLE)

    corridor_mask = make_corridor_mask(uv_gt, rgb.shape, corridor_width=corridor_width)
    centerline_mask = make_centerline_mask(uv_gt, rgb.shape, line_width=centerline_width)
    heatmap = make_gaussian_heatmap_from_corridor(corridor_mask, sigma_px=gaussian_sigma_px)

    overlay = rgb.copy()
    if corridor_mask.max() > 0:
        red = np.zeros_like(rgb)
        red[..., 0] = 255
        alpha = (corridor_mask.astype(np.float32) / 255.0)[..., None] * 0.35
        overlay = np.clip(overlay * (1 - alpha) + red * alpha, 0, 255).astype(np.uint8)

    show_debug_grid_pil(rgb, centerline_mask, corridor_mask, overlay, scale=0.75)

    visible_points = len(uv_gt)
    visible_path_length = polyline_length_px(uv_gt)
    corridor_pixels = int((corridor_mask > 0).sum())

    print(f"{title} sample_dir:", sample_dir)
    print("tfrecord_path:", tfrecord_path)
    print("camera:", cam_name_to_str(CAMERA_NAME))
    print("gt shape:", gt_xy.shape)
    print("visible uv points:", visible_points)
    print("visible path length px:", round(visible_path_length, 2))
    print("corridor pixels:", corridor_pixels)

    return {
        "sample_dir": str(sample_dir),
        "tfrecord_path": str(tfrecord_path),
        "visible_uv_points": visible_points,
        "visible_path_length_px": visible_path_length,
        "corridor_pixels": corridor_pixels,
        "uv_gt": uv_gt,
        "corridor_mask": corridor_mask,
        "centerline_mask": centerline_mask,
        "gaussian_heatmap": heatmap,
    }

In [13]:

# === Экспорт одного sample ===


def save_png(arr: np.ndarray, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if arr.dtype != np.uint8:
        raise ValueError(f"save_png expects uint8, got {arr.dtype}")
    Image.fromarray(arr).save(path)


def build_sample(
    row: dict,
    split_root: Optional[Path],
    split_name: str,
    out_root: Path,
    camera_name,
    z_vehicle: float = 0.0,
    corridor_width: int = 17,
    centerline_width: int = 3,
    gaussian_sigma_px: float = 7.0,
    min_visible_points: int = 3,
    min_visible_path_length_px: float = 25.0,
    min_corridor_pixels: int = 40,
):
    sample_dir, meta = load_meta_from_row(row, split_root=split_root)
    gt_xy = extract_gt_future_xy(sample_dir, row, meta)

    tfrecord_path, e2e = load_e2ed_record(meta, RAW_WAYMO_ROOT)
    rgb, image_obj, calib_obj = get_camera_image_and_calib(e2e.frame, camera_name)
    uv_gt, _ = project_traj_xy(gt_xy, calib_obj, rgb.shape, z_vehicle=z_vehicle)

    visible_points = len(uv_gt)
    if visible_points < min_visible_points:
        raise ValueError(
            f"Too few visible GT points on {cam_name_to_str(camera_name)}: "
            f"{visible_points} < {min_visible_points}"
        )

    path_len_px = polyline_length_px(uv_gt)
    if path_len_px < min_visible_path_length_px:
        raise ValueError(
            f"Visible GT path too short on {cam_name_to_str(camera_name)}: "
            f"{path_len_px:.2f}px < {min_visible_path_length_px:.2f}px"
        )

    corridor_mask = make_corridor_mask(uv_gt, rgb.shape, corridor_width=corridor_width)
    centerline_mask = make_centerline_mask(uv_gt, rgb.shape, line_width=centerline_width)
    gaussian_heatmap = make_gaussian_heatmap_from_corridor(corridor_mask, sigma_px=gaussian_sigma_px)

    corridor_pixels = int((corridor_mask > 0).sum())
    if corridor_pixels < min_corridor_pixels:
        raise ValueError(
            f"Corridor mask too small on {cam_name_to_str(camera_name)}: "
            f"{corridor_pixels} < {min_corridor_pixels}"
        )

    sample_id = row.get("sample_id", Path(sample_dir).name)
    cam_str = cam_name_to_str(camera_name)

    out_dir = out_root / split_name / cam_str / str(sample_id)
    out_dir.mkdir(parents=True, exist_ok=True)

    image_path = out_dir / "image.png"
    corridor_path = out_dir / "corridor_mask.png"
    centerline_path = out_dir / "centerline_mask.png"
    heatmap_path = out_dir / "gaussian_heatmap.npy"
    uv_path = out_dir / "projected_uv_gt.npy"
    meta_path = out_dir / "meta.json"

    save_png(rgb.astype(np.uint8), image_path)
    save_png(corridor_mask.astype(np.uint8), corridor_path)
    save_png(centerline_mask.astype(np.uint8), centerline_path)
    np.save(heatmap_path, gaussian_heatmap.astype(np.float32))
    np.save(uv_path, uv_gt.astype(np.float32))

    sample_meta = {
        "split": split_name,
        "sample_id": str(sample_id),
        "camera_name": cam_str,
        "sample_dir": str(sample_dir),
        "tfrecord_path": str(tfrecord_path),
        "image_path": str(image_path),
        "corridor_mask_path": str(corridor_path),
        "centerline_mask_path": str(centerline_path),
        "gaussian_heatmap_path": str(heatmap_path),
        "projected_uv_gt_path": str(uv_path),
        "gt_num_points": int(gt_xy.shape[0]),
        "visible_uv_points": int(visible_points),
        "visible_path_length_px": float(path_len_px),
        "corridor_pixels": int(corridor_pixels),
        "corridor_width_px": int(corridor_width),
        "centerline_width_px": int(centerline_width),
        "gaussian_sigma_px": float(gaussian_sigma_px),
        "min_visible_points": int(min_visible_points),
        "min_visible_path_length_px": float(min_visible_path_length_px),
        "min_corridor_pixels": int(min_corridor_pixels),
        "image_height": int(rgb.shape[0]),
        "image_width": int(rgb.shape[1]),
    }

    meta_path.write_text(json.dumps(sample_meta, ensure_ascii=False, indent=2), encoding="utf-8")
    return sample_meta


In [18]:
from pathlib import Path
from typing import List, Optional
from datetime import datetime
import json
import os
import time
import traceback
from collections import Counter


def polyline_length_px(uv: np.ndarray) -> float:
    if uv is None or len(uv) < 2:
        return 0.0
    diffs = uv[1:] - uv[:-1]
    return float(np.linalg.norm(diffs, axis=1).sum())


def _append_log_line(log_txt_path: Path, text: str, also_print: bool = False):
    log_txt_path.parent.mkdir(parents=True, exist_ok=True)
    line = text.rstrip() + "\n"
    with open(log_txt_path, "a", encoding="utf-8") as f:
        f.write(line)
        f.flush()
        os.fsync(f.fileno())
    if also_print:
        print(text, flush=True)


def _write_progress_json(progress_json_path: Path, payload: dict):
    progress_json_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = progress_json_path.with_suffix(progress_json_path.suffix + ".tmp")
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp_path, progress_json_path)


def _append_jsonl(jsonl_path: Path, payload: dict):
    jsonl_path.parent.mkdir(parents=True, exist_ok=True)
    line = json.dumps(payload, ensure_ascii=False) + "\n"
    with open(jsonl_path, "a", encoding="utf-8") as f:
        f.write(line)
        f.flush()
        os.fsync(f.fileno())


def export_split(
    rows: List[dict],
    split_name: str,
    split_root: Optional[Path],
    out_root: Path,
    camera_name,
    z_vehicle: float = 0.0,
    corridor_width: int = 17,
    centerline_width: int = 3,
    gaussian_sigma_px: float = 7.0,
    min_visible_points: int = 3,
    min_visible_path_length_px: float = 25.0,
    min_corridor_pixels: int = 40,
    limit: Optional[int] = None,
):
    cam_str = cam_name_to_str(camera_name)

    logs_dir = out_root / "_logs"
    log_txt_path = logs_dir / f"{split_name}_{cam_str}.log"
    skipped_log_path = logs_dir / f"{split_name}_{cam_str}_skipped.log"
    progress_json_path = logs_dir / f"{split_name}_{cam_str}_progress.json"
    errors_jsonl_path = logs_dir / f"{split_name}_{cam_str}_errors.jsonl"
    manifest_path = out_root / split_name / cam_str / "manifest.jsonl"
    manifest_path.parent.mkdir(parents=True, exist_ok=True)

    start_dt = datetime.now()
    start_ts = start_dt.isoformat(timespec="seconds")
    start_time = time.time()

    ok = 0
    skipped = 0
    failed = 0
    error_type_counter = Counter()

    use_rows = rows if limit is None else rows[:limit]
    total = len(use_rows)

    last_sample_id = None
    last_idx = -1
    last_progress_write_time = 0.0

    _append_log_line(
        log_txt_path,
        f"[{start_ts}] START export split={split_name} camera={cam_str} total={total}",
        also_print=True,
    )
    _append_log_line(
        log_txt_path,
        f"[{start_ts}] out_root={out_root}",
    )
    _append_log_line(
        log_txt_path,
        f"[{start_ts}] manifest_path={manifest_path}",
    )
    _append_log_line(
        log_txt_path,
        f"[{start_ts}] errors_jsonl_path={errors_jsonl_path}",
    )

    progress_payload = {
        "status": "running",
        "split": split_name,
        "camera": cam_str,
        "started_at": start_ts,
        "updated_at": start_ts,
        "processed": 0,
        "total": total,
        "ok": 0,
        "skipped": 0,
        "failed": 0,
        "keep_rate_percent": 0.0,
        "last_idx": None,
        "last_sample_id": None,
        "elapsed_sec": 0.0,
        "samples_per_sec": 0.0,
        "manifest_path": str(manifest_path),
        "log_txt_path": str(log_txt_path),
        "skipped_log_path": str(skipped_log_path),
        "errors_jsonl_path": str(errors_jsonl_path),
        "error_type_counts": {},
    }
    _write_progress_json(progress_json_path, progress_payload)

    try:
        with open(manifest_path, "w", encoding="utf-8") as manifest_f:
            for idx, row in enumerate(use_rows):
                last_idx = idx
                last_sample_id = row.get("sample_id")
                row_start_time = time.time()

                try:
                    meta_out = build_sample(
                        row=row,
                        split_root=split_root,
                        split_name=split_name,
                        out_root=out_root,
                        camera_name=camera_name,
                        z_vehicle=z_vehicle,
                        corridor_width=corridor_width,
                        centerline_width=centerline_width,
                        gaussian_sigma_px=gaussian_sigma_px,
                        min_visible_points=min_visible_points,
                        min_visible_path_length_px=min_visible_path_length_px,
                        min_corridor_pixels=min_corridor_pixels,
                    )
                    manifest_f.write(json.dumps(meta_out, ensure_ascii=False) + "\n")
                    manifest_f.flush()
                    os.fsync(manifest_f.fileno())
                    ok += 1

                except ValueError as e:
                    skipped += 1
                    msg = (
                        f"[SKIPPED] split={split_name} idx={idx} "
                        f"sample_id={last_sample_id} :: {e}"
                    )
                    if skipped <= 50:
                        print(msg, flush=True)
                    _append_log_line(log_txt_path, msg)
                    _append_log_line(skipped_log_path, msg)

                except Exception as e:
                    failed += 1
                    err_type = type(e).__name__
                    error_type_counter[err_type] += 1

                    msg = (
                        f"[FAILED] split={split_name} idx={idx} "
                        f"sample_id={last_sample_id} :: {err_type}: {e}"
                    )
                    if failed <= 50:
                        print(msg, flush=True)

                    _append_log_line(log_txt_path, msg)

                    _append_jsonl(
                        errors_jsonl_path,
                        {
                            "ts": datetime.now().isoformat(timespec="seconds"),
                            "split": split_name,
                            "camera": cam_str,
                            "idx": idx,
                            "sample_id": last_sample_id,
                            "error_type": err_type,
                            "error": str(e),
                            "traceback": traceback.format_exc(),
                        },
                    )

                processed = idx + 1
                now = time.time()
                elapsed = now - start_time
                row_elapsed = now - row_start_time

                need_progress_write = (
                    processed % 200 == 0
                    or processed == total
                    or (now - last_progress_write_time) >= 30.0
                )

                if need_progress_write:
                    sps = processed / elapsed if elapsed > 0 else 0.0
                    keep_rate = round(100.0 * ok / processed, 4) if processed > 0 else 0.0

                    line = (
                        f"processed {processed}/{total} | ok={ok} skipped={skipped} failed={failed} "
                        f"| last_idx={idx} sample_id={last_sample_id} "
                        f"| last_row_sec={row_elapsed:.3f} total_sec={elapsed:.1f} sps={sps:.3f}"
                    )
                    print(line, flush=True)
                    _append_log_line(log_txt_path, line)

                    progress_payload = {
                        "status": "running" if processed < total else "finished",
                        "split": split_name,
                        "camera": cam_str,
                        "started_at": start_ts,
                        "updated_at": datetime.now().isoformat(timespec="seconds"),
                        "processed": processed,
                        "total": total,
                        "ok": ok,
                        "skipped": skipped,
                        "failed": failed,
                        "keep_rate_percent": keep_rate,
                        "last_idx": idx,
                        "last_sample_id": last_sample_id,
                        "elapsed_sec": round(elapsed, 3),
                        "samples_per_sec": round(sps, 6),
                        "manifest_path": str(manifest_path),
                        "log_txt_path": str(log_txt_path),
                        "skipped_log_path": str(skipped_log_path),
                        "errors_jsonl_path": str(errors_jsonl_path),
                        "error_type_counts": dict(error_type_counter),
                    }
                    _write_progress_json(progress_json_path, progress_payload)
                    last_progress_write_time = now

    except Exception as e:
        fatal_ts = datetime.now().isoformat(timespec="seconds")
        fatal_type = type(e).__name__
        fatal_msg = (
            f"[FATAL] export_split crashed :: split={split_name} camera={cam_str} "
            f"last_idx={last_idx} sample_id={last_sample_id} :: {fatal_type}: {e}"
        )

        print(fatal_msg, flush=True)
        _append_log_line(log_txt_path, f"[{fatal_ts}] {fatal_msg}")

        _append_jsonl(
            errors_jsonl_path,
            {
                "ts": fatal_ts,
                "split": split_name,
                "camera": cam_str,
                "idx": last_idx,
                "sample_id": last_sample_id,
                "error_type": fatal_type,
                "error": str(e),
                "traceback": traceback.format_exc(),
                "fatal": True,
            },
        )

        elapsed = time.time() - start_time
        progress_payload = {
            "status": "crashed",
            "split": split_name,
            "camera": cam_str,
            "started_at": start_ts,
            "updated_at": fatal_ts,
            "processed": max(last_idx + 1, 0),
            "total": total,
            "ok": ok,
            "skipped": skipped,
            "failed": failed,
            "keep_rate_percent": round(100.0 * ok / max(last_idx + 1, 1), 4) if last_idx >= 0 else 0.0,
            "last_idx": last_idx,
            "last_sample_id": last_sample_id,
            "elapsed_sec": round(elapsed, 3),
            "samples_per_sec": round((max(last_idx + 1, 0) / elapsed), 6) if elapsed > 0 else 0.0,
            "manifest_path": str(manifest_path),
            "log_txt_path": str(log_txt_path),
            "skipped_log_path": str(skipped_log_path),
            "errors_jsonl_path": str(errors_jsonl_path),
            "error_type_counts": dict(error_type_counter),
            "fatal_error_type": fatal_type,
            "fatal_error": str(e),
        }
        _write_progress_json(progress_json_path, progress_payload)
        raise

    end_dt = datetime.now()
    end_ts = end_dt.isoformat(timespec="seconds")
    elapsed = time.time() - start_time
    sps = total / elapsed if elapsed > 0 else 0.0

    final_line = (
        f"done split={split_name}: ok={ok} skipped={skipped} failed={failed} "
        f"| total={total} | elapsed_sec={elapsed:.1f} | sps={sps:.3f}"
    )

    print(final_line, flush=True)
    print(f"manifest saved to: {manifest_path}", flush=True)
    print(f"progress log: {log_txt_path}", flush=True)
    print(f"skipped log: {skipped_log_path}", flush=True)
    print(f"progress json: {progress_json_path}", flush=True)
    print(f"errors jsonl: {errors_jsonl_path}", flush=True)

    _append_log_line(log_txt_path, f"[{end_ts}] {final_line}")
    _append_log_line(log_txt_path, f"manifest saved to: {manifest_path}")
    _append_log_line(log_txt_path, f"skipped log: {skipped_log_path}")
    _append_log_line(log_txt_path, f"progress json: {progress_json_path}")
    _append_log_line(log_txt_path, f"errors jsonl: {errors_jsonl_path}")

    progress_payload = {
        "status": "finished",
        "split": split_name,
        "camera": cam_str,
        "started_at": start_ts,
        "finished_at": end_ts,
        "processed": total,
        "total": total,
        "ok": ok,
        "skipped": skipped,
        "failed": failed,
        "keep_rate_percent": round(100.0 * ok / total, 4) if total > 0 else 0.0,
        "last_idx": total - 1 if total > 0 else None,
        "last_sample_id": last_sample_id,
        "elapsed_sec": round(elapsed, 3),
        "samples_per_sec": round(sps, 6),
        "manifest_path": str(manifest_path),
        "log_txt_path": str(log_txt_path),
        "skipped_log_path": str(skipped_log_path),
        "errors_jsonl_path": str(errors_jsonl_path),
        "error_type_counts": dict(error_type_counter),
    }
    _write_progress_json(progress_json_path, progress_payload)

    return progress_payload

In [19]:
# === Рекомендуемый порядок запуска ===
# 1) Сначала вручную запусти debug_one_row(train_rows[0], ...) и debug_one_row(val_rows[0], ...).
# 2) Потом запусти export_split для train и val.
# 3) Во время выполнения смотри прогресс в отдельных файлах:
#    prepared_camera_corridor_dataset_filtered_front_v1/_logs/train_FRONT.log
#    prepared_camera_corridor_dataset_filtered_front_v1/_logs/train_FRONT_progress.json
#    prepared_camera_corridor_dataset_filtered_front_v1/_logs/val_FRONT.log
#    prepared_camera_corridor_dataset_filtered_front_v1/_logs/val_FRONT_progress.json
# 4) Готовые результаты ищи здесь:
#    prepared_camera_corridor_dataset_filtered_front_v1/train/FRONT/manifest.jsonl
#    prepared_camera_corridor_dataset_filtered_front_v1/val/FRONT/manifest.jsonl

# Пример ручной проверки одного sample:
# debug_one_row(train_rows[0], split_root=TRAIN_ROOT, title="TRAIN")
# debug_one_row(val_rows[0], split_root=VAL_ROOT, title="VAL")


In [20]:
import inspect
print(inspect.signature(export_split))

(rows: List[dict], split_name: str, split_root: Optional[pathlib.Path], out_root: pathlib.Path, camera_name, z_vehicle: float = 0.0, corridor_width: int = 17, centerline_width: int = 3, gaussian_sigma_px: float = 7.0, min_visible_points: int = 3, min_visible_path_length_px: float = 25.0, min_corridor_pixels: int = 40, limit: Optional[int] = None)



## Что именно предсказывать дальше

На первом шаге разумно учить модель на **corridor_mask**.

Почему:
- она устойчивее тонкой линии;
- лучше переносит мелкие ошибки проекции;
- ближе к идее «области допустимого движения».

Дополнительно уже сохранены:
- `centerline_mask` — как более жесткий target;
- `gaussian_heatmap` — как мягкий target.

Дальше можно сравнить три режима обучения:
1. `RGB -> corridor_mask`
2. `RGB + route prior -> corridor_mask`
3. `RGB + route prior -> gaussian_heatmap`


In [ ]:
manifest_train = export_split(
    rows=train_rows,
    split_name="train",
    split_root=TRAIN_ROOT,
    out_root=OUT_ROOT,
    camera_name=CAMERA_NAME,
    z_vehicle=Z_VEHICLE,
    corridor_width=CORRIDOR_WIDTH,
    centerline_width=CENTERLINE_WIDTH,
    gaussian_sigma_px=GAUSSIAN_SIGMA_PX,
    min_visible_points=MIN_VISIBLE_POINTS,
    min_visible_path_length_px=MIN_VISIBLE_PATH_LENGTH_PX,
    min_corridor_pixels=MIN_CORRIDOR_PIXELS,
)

manifest_val = export_split(
    rows=val_rows,
    split_name="val",
    split_root=VAL_ROOT,
    out_root=OUT_ROOT,
    camera_name=CAMERA_NAME,
    z_vehicle=Z_VEHICLE,
    corridor_width=CORRIDOR_WIDTH,
    centerline_width=CENTERLINE_WIDTH,
    gaussian_sigma_px=GAUSSIAN_SIGMA_PX,
    min_visible_points=MIN_VISIBLE_POINTS,
    min_visible_path_length_px=MIN_VISIBLE_PATH_LENGTH_PX,
    min_corridor_pixels=MIN_CORRIDOR_PIXELS,
)

[2026-04-21T22:13:10] START export split=train camera=FRONT total=415663
processed 1/415663 | ok=1 skipped=0 failed=0 | last_idx=0 sample_id=0 | last_row_sec=0.503 total_sec=0.5 sps=1.965
[SKIPPED] split=train idx=1 sample_id=1 :: Too few visible GT points on FRONT: 0 < 3
[SKIPPED] split=train idx=2 sample_id=2 :: Too few visible GT points on FRONT: 0 < 3
[SKIPPED] split=train idx=8 sample_id=8 :: Too few visible GT points on FRONT: 0 < 3
[SKIPPED] split=train idx=10 sample_id=10 :: Too few visible GT points on FRONT: 0 < 3
[SKIPPED] split=train idx=16 sample_id=16 :: Too few visible GT points on FRONT: 1 < 3
[SKIPPED] split=train idx=26 sample_id=26 :: Too few visible GT points on FRONT: 2 < 3
[SKIPPED] split=train idx=30 sample_id=30 :: Too few visible GT points on FRONT: 1 < 3
[SKIPPED] split=train idx=32 sample_id=32 :: Too few visible GT points on FRONT: 0 < 3


In [ ]:
import inspect
print(inspect.signature(export_split))

In [ ]:
# === Где смотреть результаты после экспорта ===
print('OUT_ROOT =', OUT_ROOT.resolve())
print('train manifest =', (OUT_ROOT / 'train' / 'FRONT' / 'manifest.jsonl').resolve())
print('val manifest   =', (OUT_ROOT / 'val' / 'FRONT' / 'manifest.jsonl').resolve())
print('train log      =', (OUT_ROOT / '_logs' / 'train_FRONT.log').resolve())
print('val log        =', (OUT_ROOT / '_logs' / 'val_FRONT.log').resolve())
print('train progress =', (OUT_ROOT / '_logs' / 'train_FRONT_progress.json').resolve())
print('val progress   =', (OUT_ROOT / '_logs' / 'val_FRONT_progress.json').resolve())